In [1]:
import torch
import os 
import json
import torch_geometric
import re
import gc
import wandb
import optuna
import warnings
import time

from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler
from torch_geometric.data import Data, HeteroData, Batch
from torch_geometric.loader import DataLoader
from torch_geometric.utils import to_dense_adj
from torch_geometric.nn import to_hetero
from collections import defaultdict, Counter
from tqdm import tqdm
from sklearn.metrics import ndcg_score
from itertools import groupby, permutations
from transformers import AutoTokenizer, AutoModel
from optuna.integration.wandb import WeightsAndBiasesCallback

import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import networkx as nx
import numpy as np
import pandas as pd
import torch.nn as nn
import torch_geometric.nn as geom_nn
import torch_geometric.data as geom_data

C:\Users\roans\anaconda3.0\envs\transformers\lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
# different metric (also ndcg@20)+
# add more randomo 0s

In [3]:
torch_geometric.__version__, torch.__version__

('2.7.0', '2.6.0+cu124')

In [4]:
device = ("cuda:0" if torch.cuda.is_available() else "cpu")
device, torch.cuda.get_device_name(0)

('cuda:0', 'NVIDIA GeForce RTX 4070 Ti')

In [5]:
import copy
import warnings
from typing import Any, Dict, List, Optional, Union

import torch
from torch import Tensor
from torch.nn import Module, Parameter

from torch_geometric.nn.conv import MessagePassing
from torch_geometric.nn.dense import Linear
from torch_geometric.nn.fx import Transformer
from torch_geometric.typing import EdgeType, Metadata, NodeType, SparseTensor
from torch_geometric.utils.hetero import get_unused_node_types

try:
    from torch.fx import Graph, GraphModule, Node
except (ImportError, ModuleNotFoundError, AttributeError):
    GraphModule, Graph, Node = 'GraphModule', 'Graph', 'Node'


def to_hetero_with_bases(module: Module, metadata: Metadata, num_bases: int,
                         in_channels: Optional[Dict[str, int]] = None,
                         input_map: Optional[Dict[str, str]] = None,
                         debug: bool = False) -> GraphModule:

    transformer = ToHeteroWithBasesTransformer(module, metadata, num_bases,
                                               in_channels, input_map, debug)
    return transformer.transform()



class ToHeteroWithBasesTransformer(Transformer):
    def __init__(
        self,
        module: Module,
        metadata: Metadata,
        num_bases: int,
        in_channels: Optional[Dict[str, int]] = None,
        input_map: Optional[Dict[str, str]] = None,
        debug: bool = False,
    ):
        super().__init__(module, input_map, debug)

        self.metadata = metadata
        self.num_bases = num_bases
        self.in_channels = in_channels or {}
        assert len(metadata) == 2
        assert len(metadata[0]) > 0 and len(metadata[1]) > 0

        self.validate()

        # Compute IDs for each node and edge type:
        self.node_type2id = {k: i for i, k in enumerate(metadata[0])}
        self.edge_type2id = {k: i for i, k in enumerate(metadata[1])}

    def validate(self):
        unused_node_types = get_unused_node_types(*self.metadata)
        if len(unused_node_types) > 0:
            warnings.warn(
                f"There exist node types ({unused_node_types}) whose "
                f"representations do not get updated during message passing "
                f"as they do not occur as destination type in any edge type. "
                f"This may lead to unexpected behavior.")

        names = self.metadata[0] + [rel for _, rel, _ in self.metadata[1]]
        for name in names:
            if not name.isidentifier():
                warnings.warn(
                    f"The type '{name}' contains invalid characters which "
                    f"may lead to unexpected behavior. To avoid any issues, "
                    f"ensure that your types only contain letters, numbers "
                    f"and underscores.")

    def transform(self) -> GraphModule:
        self._node_offset_dict_initialized = False
        self._edge_offset_dict_initialized = False
        self._edge_type_initialized = False
        out = super().transform()
        del self._node_offset_dict_initialized
        del self._edge_offset_dict_initialized
        del self._edge_type_initialized
        return out

    def placeholder(self, node: Node, target: Any, name: str):
        if node.type is not None:
            Type = EdgeType if self.is_edge_level(node) else NodeType
            node.type = Dict[Type, node.type]

        out = node

        # Create `node_offset_dict` and `edge_offset_dict` dictionaries in case
        # they are not yet initialized. These dictionaries hold the cumulated
        # sizes used to create a unified graph representation and to split the
        # output data.
        if self.is_edge_level(node) and not self._edge_offset_dict_initialized:
            self.graph.inserting_after(out)
            out = self.graph.create_node('call_function',
                                         target=get_edge_offset_dict,
                                         args=(node, self.edge_type2id),
                                         name='edge_offset_dict')
            self._edge_offset_dict_initialized = True

        elif not self._node_offset_dict_initialized:
            self.graph.inserting_after(out)
            out = self.graph.create_node('call_function',
                                         target=get_node_offset_dict,
                                         args=(node, self.node_type2id),
                                         name='node_offset_dict')
            self._node_offset_dict_initialized = True

        # Create a `edge_type` tensor used as input to `HeteroBasisConv`:
        if self.is_edge_level(node) and not self._edge_type_initialized:
            self.graph.inserting_after(out)
            out = self.graph.create_node('call_function', target=get_edge_type,
                                         args=(node, self.edge_type2id),
                                         name='edge_type')
            self._edge_type_initialized = True

        # Add `Linear` operation to align features to the same dimensionality:
        if name in self.in_channels:
            self.graph.inserting_after(out)
            out = self.graph.create_node('call_module',
                                         target=f'align_lin__{name}',
                                         args=(node, ),
                                         name=f'{name}__aligned')
            self._state[out.name] = self._state[name]

            lin = LinearAlign(self.metadata[int(self.is_edge_level(node))],
                              self.in_channels[name])
            setattr(self.module, f'align_lin__{name}', lin)

        # Perform grouping of type-wise values into a single tensor:
        if self.is_edge_level(node):
            self.graph.inserting_after(out)
            out = self.graph.create_node(
                'call_function', target=group_edge_placeholder,
                args=(out if name in self.in_channels else node,
                      self.edge_type2id,
                      self.find_by_name('node_offset_dict')),
                name=f'{name}__grouped')
            self._state[out.name] = 'edge'

        else:
            self.graph.inserting_after(out)
            out = self.graph.create_node(
                'call_function', target=group_node_placeholder,
                args=(out if name in self.in_channels else node,
                      self.node_type2id), name=f'{name}__grouped')
            self._state[out.name] = 'node'

        self.replace_all_uses_with(node, out)

    def call_message_passing_module(self, node: Node, target: Any, name: str):
        # Call the `HeteroBasisConv` wrapper instead instead of a single
        # message passing layer. We need to inject the `edge_type` as first
        # argument in order to do so.
        node.args = (self.find_by_name('edge_type'), ) + node.args

    def output(self, node: Node, target: Any, name: str):
        # Split the output to dictionaries, holding either node type-wise or
        # edge type-wise data.
        def _recurse(value: Any) -> Any:
            if isinstance(value, Node) and self.is_edge_level(value):
                self.graph.inserting_before(node)
                return self.graph.create_node(
                    'call_function', target=split_output,
                    args=(value, self.find_by_name('edge_offset_dict')),
                    name=f'{value.name}__split')

                pass
            elif isinstance(value, Node):
                self.graph.inserting_before(node)
                return self.graph.create_node(
                    'call_function', target=split_output,
                    args=(value, self.find_by_name('node_offset_dict')),
                    name=f'{value.name}__split')

            elif isinstance(value, dict):
                return {k: _recurse(v) for k, v in value.items()}
            elif isinstance(value, list):
                return [_recurse(v) for v in value]
            elif isinstance(value, tuple):
                return tuple(_recurse(v) for v in value)
            else:
                return value

        if node.type is not None and isinstance(node.args[0], Node):
            output = node.args[0]
            Type = EdgeType if self.is_edge_level(output) else NodeType
            node.type = Dict[Type, node.type]
        else:
            node.type = None

        node.args = (_recurse(node.args[0]), )

    def init_submodule(self, module: Module, target: str) -> Module:
        if not isinstance(module, MessagePassing):
            return module

        # Replace each `MessagePassing` module by a `HeteroBasisConv` wrapper:
        return HeteroBasisConv(module, len(self.metadata[1]), self.num_bases)


###############################################################################


class HeteroBasisConv(torch.nn.Module):
    # A wrapper layer that applies the basis-decomposition technique to a
    # heterogeneous graph.
    def __init__(self, module: MessagePassing, num_relations: int,
                 num_bases: int):
        super().__init__()

        self.num_relations = num_relations
        self.num_bases = num_bases

        # We make use of a post-message computation hook to inject the
        # basis re-weighting for each individual edge type.
        # This currently requires us to set `conv.fuse = False`, which leads
        # to a materialization of messages.
        def hook(module, inputs, output):
            assert isinstance(module._edge_type, Tensor)
            if module._edge_type.size(0) != output.size(0):
                raise ValueError(
                    f"Number of messages ({output.size(0)}) does not match "
                    f"with the number of original edges "
                    f"({module._edge_type.size(0)}). Does your message "
                    f"passing layer create additional self-loops? Try to "
                    f"remove them via 'add_self_loops=False'")
            weight = module.edge_type_weight.view(-1)[module._edge_type]
            weight = weight.view([-1] + [1] * (output.dim() - 1))
            return weight * output

        params = list(module.parameters())
        device = params[0].device if len(params) > 0 else 'cpu'

        self.convs = torch.nn.ModuleList()
        for _ in range(num_bases):
            conv = copy.deepcopy(module)
            conv.fuse = False  # Disable `message_and_aggregate` functionality.
            # We learn a single scalar weight for each individual edge type,
            # which is used to weight the output message based on edge type:
            conv.edge_type_weight = Parameter(
                torch.empty(1, num_relations, device=device))
            conv.register_message_forward_hook(hook)
            self.convs.append(conv)

        if self.num_bases > 1:
            self.reset_parameters()

    def reset_parameters(self):
        for conv in self.convs:
            if hasattr(conv, 'reset_parameters'):
                conv.reset_parameters()
            elif sum([p.numel() for p in conv.parameters()]) > 0:
                warnings.warn(
                    f"'{conv}' will be duplicated, but its parameters cannot "
                    f"be reset. To suppress this warning, add a "
                    f"'reset_parameters()' method to '{conv}'")
            torch.nn.init.xavier_uniform_(conv.edge_type_weight)

    def forward(self, edge_type: Tensor, *args, **kwargs) -> Tensor:
        out = None
        
        attention = []
        
        # Call message passing modules and perform aggregation:
        for conv in self.convs:
            conv._edge_type = edge_type
                        
            # res, (edge_ind_exp, att_weight_exp) = conv(*args, **kwargs)
            res = conv(*args, **kwargs)
            del conv._edge_type
            
            # attention.append(att_weight_exp)
            
            out = res if out is None else out.add_(res)
            
            # jump
        
        return out #, (edge_type, edge_ind_exp, torch.mean(torch.stack(attention, dim=0), dim=0))

    def __repr__(self) -> str:
        return (f'{self.__class__.__name__}(num_relations='
                f'{self.num_relations}, num_bases={self.num_bases})')


class LinearAlign(torch.nn.Module):
    # Aligns representions to the same dimensionality. Note that this will
    # create lazy modules, and as such requires a forward pass in order to
    # initialize parameters.
    def __init__(self, keys: List[Union[NodeType, EdgeType]],
                 out_channels: int):
        super().__init__()
        self.out_channels = out_channels
        self.lins = torch.nn.ModuleDict()
        for key in keys:
            self.lins[key2str(key)] = Linear(-1, out_channels, bias=False)

    def forward(
        self, x_dict: Dict[Union[NodeType, EdgeType], Tensor]
    ) -> Dict[Union[NodeType, EdgeType], Tensor]:
        
        return {key: self.lins[key2str(key)](x) for key, x in x_dict.items()}

    def __repr__(self) -> str:
        return (f'{self.__class__.__name__}(num_relations={len(self.lins)}, '
                f'out_channels={self.out_channels})')


###############################################################################

# These methods are used in order to receive the cumulated sizes of input
# dictionaries. We make use of them for creating a unified homogeneous graph
# representation, as well as to split the final output data once again.


def get_node_offset_dict(
    input_dict: Dict[NodeType, Union[Tensor, SparseTensor]],
    type2id: Dict[NodeType, int],
) -> Dict[NodeType, int]:
    cumsum = 0
    out: Dict[NodeType, int] = {}
    
    for key in type2id.keys():
        out[key] = cumsum
        cumsum += input_dict[key].size(0)

    return out


def get_edge_offset_dict(
    input_dict: Dict[EdgeType, Union[Tensor, SparseTensor]],
    type2id: Dict[EdgeType, int],
) -> Dict[EdgeType, int]:
    cumsum = 0
    out: Dict[EdgeType, int] = {}
    for key in type2id.keys():
        out[key] = cumsum
        value = input_dict[key]
        if isinstance(value, SparseTensor):
            cumsum += value.nnz()
        elif value.dtype == torch.long and value.size(0) == 2:
            cumsum += value.size(-1)
        else:
            cumsum += value.size(0)

    return out


###############################################################################

# This method computes the edge type of the final homogeneous graph
# representation. It will be used in the `HeteroBasisConv` wrapper.


def get_edge_type(
    input_dict: Dict[EdgeType, Union[Tensor, SparseTensor]],
    type2id: Dict[EdgeType, int],
) -> Tensor:

    inputs = [input_dict[key] for key in type2id.keys()]
    outs = []

    for i, value in enumerate(inputs):
        if value.size(0) == 2 and value.dtype == torch.long:  # edge_index
            out = value.new_full((value.size(-1), ), i, dtype=torch.long)
        elif isinstance(value, SparseTensor):
            out = torch.full((value.nnz(), ), i, dtype=torch.long,
                             device=value.device())
        else:
            out = value.new_full((value.size(0), ), i, dtype=torch.long)
        outs.append(out)
    
    return outs[0] if len(outs) == 1 else torch.cat(outs, dim=0)


###############################################################################

# These methods are used to group the individual type-wise components into a
# unfied single representation.


def group_node_placeholder(input_dict: Dict[NodeType, Tensor],
                           type2id: Dict[NodeType, int]) -> Tensor:

    inputs = [input_dict[key] for key in type2id.keys()]
    return inputs[0] if len(inputs) == 1 else torch.cat(inputs, dim=0)


def group_edge_placeholder(
    input_dict: Dict[EdgeType, Union[Tensor, SparseTensor]],
    type2id: Dict[EdgeType, int],
    offset_dict: Dict[NodeType, int] = None,
) -> Union[Tensor, SparseTensor]:

    inputs = [input_dict[key] for key in type2id.keys()]

    if len(inputs) == 1:
        return inputs[0]

    # In case of grouping a graph connectivity tensor `edge_index` or `adj_t`,
    # we need to increment its indices:
    elif inputs[0].size(0) == 2 and inputs[0].dtype == torch.long:
        if offset_dict is None:
            raise AttributeError(
                "Can not infer node-level offsets. Please ensure that there "
                "exists a node-level argument before the 'edge_index' "
                "argument in your forward header.")

        outputs = []
        for value, (src_type, _, dst_type) in zip(inputs, type2id):
            value = value.clone()
            value[0, :] += offset_dict[src_type]
            value[1, :] += offset_dict[dst_type]
            outputs.append(value)

        return torch.cat(outputs, dim=-1)

    elif isinstance(inputs[0], SparseTensor):
        if offset_dict is None:
            raise AttributeError(
                "Can not infer node-level offsets. Please ensure that there "
                "exists a node-level argument before the 'SparseTensor' "
                "argument in your forward header.")

        # For grouping a list of SparseTensors, we convert them into a
        # unified `edge_index` representation in order to avoid conflicts
        # induced by re-shuffling the data.
        rows, cols = [], []
        for value, (src_type, _, dst_type) in zip(inputs, type2id):
            col, row, value = value.coo()
            assert value is None
            rows.append(row + offset_dict[src_type])
            cols.append(col + offset_dict[dst_type])

        row = torch.cat(rows, dim=0)
        col = torch.cat(cols, dim=0)
        return torch.stack([row, col], dim=0)

    else:
        return torch.cat(inputs, dim=0)


###############################################################################

# This method is used to split the output tensors into individual type-wise
# components:


def split_output(
    output: Tensor,
    offset_dict: Union[Dict[NodeType, int], Dict[EdgeType, int]],
) -> Union[Dict[NodeType, Tensor], Dict[EdgeType, Tensor]]:
    
    # Sometimes an edge index ends up here. Not sure why. TODO: fix --> we should be able to determine which edge belongs
    # to which edge type
    if type(output) == tuple:
        return output
    elif output.size(0) == 2:
        output = output.T
        
    cumsums = list(offset_dict.values()) + [output.size(0)]    
    sizes = [cumsums[i + 1] - cumsums[i] for i in range(len(offset_dict))]
    outputs = output.split(sizes)
    
    return {key: output for key, output in zip(offset_dict, outputs)}


###############################################################################


def key2str(key: Union[NodeType, EdgeType]) -> str:
    key = '__'.join(key) if isinstance(key, tuple) else key
    return key.replace(' ', '_').replace('-', '_').replace(':', '_')

In [6]:
def listwise_loss(scores, labels):
    if labels.size(0) < 2:
        return torch.zeros((labels.size(0), 1), device=scores.device)

    # 1. Expand scores and labels into [N, N] matrices
    # S_i[i, j] is score of doc i, S_j[i, j] is score of doc j
    s_i = scores.view(-1, 1)
    s_j = scores.view(1, -1)
    l_i = labels.view(-1, 1)
    l_j = labels.view(1, -1)

    # 2. Only compute loss for pairs where label_i > label_j
    # This removes the "Intra-label Noise"
    pair_mask = (l_i > l_j).float()
    
    # 3. Calculate RankNet Gradient: sigmoid(s_j - s_i)
    # Using the property: 1 / (1 + exp(s_i - s_j)) = sigmoid(s_j - s_i)
    sigma = 1.0
    lambda_ij = torch.sigmoid(sigma * (s_j - s_i))

    # 4. Calculate Delta-NDCG
    # We sort to get the ranks
    sorted_idx = torch.argsort(scores.view(-1), descending=True)
    ranks = torch.zeros_like(sorted_idx)
    ranks[sorted_idx] = torch.arange(len(scores), device=scores.device)
    
    # Ranks for i and j
    r_i = ranks.view(-1, 1)
    r_j = ranks.view(1, -1)
    
    # Ideal DCG for normalization
    ideal_labels, _ = torch.sort(labels, descending=True)
    k = torch.arange(1, len(labels) + 1, device=scores.device)
    idcg = torch.sum((2**ideal_labels - 1) / torch.log2(k + 1))
    
    if idcg == 0: return torch.zeros_like(scores)

    # Calculate how much NDCG would change if we swapped i and j
    # (2^li - 2^lj) * (1/log(ri+1) - 1/log(rj+1))
    gain_diff = (2**l_i - 2**l_j)
    decay_diff = (1.0 / torch.log2(r_i + 2.0) - 1.0 / torch.log2(r_j + 2.0)).abs()
    delta_ndcg = (gain_diff * decay_diff) / idcg

    # 5. Aggregate Lambdas
    # Total force on doc i is the sum of all pairs where i is better than j
    # and all pairs where j is better than i (with flipped sign)
    # Force on i = sum_j (lambda_ij * delta_ndcg) where l_i > l_j
    # minus sum_j (lambda_ji * delta_ndcg) where l_j > l_i
    
    # This simplifies to:
    force_matrix = lambda_ij * delta_ndcg * pair_mask
    lambda_i = -torch.sum(force_matrix, dim=1) + torch.sum(force_matrix, dim=0)
    
    return lambda_i.view(-1, 1)

In [7]:
class InitialTransformLayer(torch.nn.Module):
    def __init__(self, embedding_size=32):
        super().__init__()        
        self.lin = nn.LazyLinear(embedding_size) 
        self.relu = nn.ReLU()

    # Add 'edge_index' here so it matches the call signature
    def forward(self, x, edge_index=None): 
        # We ignore edge_index to ensure it never returns 'None'
        return self.relu(self.lin(x))

class GNN(torch.nn.Module):
    def __init__(self, embedding_size=64, heads=4):
        super().__init__()
        self.pre_conv = geom_nn.TransformerConv(embedding_size, embedding_size)
        
        self.can_pos = geom_nn.GATv2Conv((-1, -1), out_channels=embedding_size, heads=heads, concat=False, add_self_loops=False)
        self.can_neg = geom_nn.GATv2Conv((-1, -1), out_channels=embedding_size, heads=heads, concat=False, add_self_loops=False)
        self.com_pos = geom_nn.GATv2Conv((-1, -1), out_channels=embedding_size, heads=heads, concat=False, add_self_loops=False)
        self.com_neg = geom_nn.GATv2Conv((-1, -1), out_channels=embedding_size, heads=heads, concat=False, add_self_loops=False)
        
        self.batch_norm = torch.nn.ModuleList([torch.nn.BatchNorm1d(embedding_size) for _ in range(4)])
        self.elu = nn.ELU()
        
    def forward(self, x, edge_index):            
        # Initial Message Passing (Basis-friendly)
        x = self.pre_conv(x, edge_index)
        
        # Multi-View logic
        x_can = self.can_neg(self.can_pos(x, edge_index.long()), edge_index.long())
        x_com = self.com_neg(self.com_pos(x * -1, edge_index[[1,0]].long()), edge_index[[1,0]].long())
        
        x_can = self.batch_norm[1](self.batch_norm[0](x_can))
        x_com = self.batch_norm[3](self.batch_norm[2](x_com))
         
        x_can, x_com = self.elu(x_can), self.elu(x_com)
        return torch.cat([x_can, x_com], dim=-1)

# Final Heterogeneous Wrapper
class OKRA(torch.nn.Module):
    def __init__(self, metadata, embedding_size=64, pooling_method="mean", heads=4):
        super().__init__()
        
        self.metadata = metadata
        self.num_heads = heads
        self.embedding_size = embedding_size
        
        self.pooling = {
            "mean": lambda x, dim: torch.mean(x, dim=dim),
            "sum": lambda x, dim: torch.sum(x, dim=dim),
            "max": lambda x, dim: torch.max(x, dim=dim)[0]
        }[pooling_method]
        
        # Initial transformation
        self.embedder = InitialTransformLayer(embedding_size=embedding_size)
        self.embedder = to_hetero(self.embedder, metadata, aggr='sum')

        self.gnn = GNN(embedding_size=embedding_size, heads=heads)
        self.gnn = to_hetero_with_bases(self.gnn, metadata, num_bases=3)
        
        # Adjust MLP size: (2 core nodes + 1 pooled context) * embedding_size
        self.mlp_candidate = nn.Linear(embedding_size * 3, 1)
        self.mlp_company = nn.Linear(embedding_size * 3, 1)

        self.sigmoid = nn.Sigmoid()
        
    def forward(self, data):
        # Device and Dtype setup
        ref_key = next(iter(data.x_dict))
        device = data.x_dict[ref_key].device
        dtype = torch.float32

        # Heal input gaps
        initial_x = {}
        for ntype in self.metadata[0]:
            if ntype in data.x_dict and data.x_dict[ntype] is not None:
                initial_x[ntype] = data.x_dict[ntype].to(dtype)
            else:
                initial_x[ntype] = torch.zeros((1, 32), device=device, dtype=dtype)

        # Heal edge gaps
        safe_edge_dict = {}
        for triplet in self.metadata[1]:
            if triplet in data.edge_index_dict:
                safe_edge_dict[triplet] = data.edge_index_dict[triplet]
            else:
                # Provide empty indices so the GNN doesn't KeyError
                safe_edge_dict[triplet] = torch.empty((2, 0), device=device, dtype=torch.long)

        # Embed
        embedded_dict = self.embedder(initial_x, safe_edge_dict)       
        gnn_out = self.gnn(embedded_dict, safe_edge_dict)       

        # Split and pool 
        x_can_dict, x_com_dict = {}, {}
        for ntype, val in gnn_out.items():
            if val is not None:
                x_can_dict[ntype], x_com_dict[ntype] = torch.chunk(val, 2, dim=-1)
            else:
                # Safety fallback for types that weren't updated by GNN
                num_nodes = initial_x[ntype].size(0)
                zeros = torch.zeros((num_nodes, self.embedding_size), device=device)
                x_can_dict[ntype], x_com_dict[ntype] = zeros, zeros

        sub_graphs_can, sub_graphs_com = defaultdict(list), defaultdict(list)
        main_nodes_can, main_nodes_com = defaultdict(list), defaultdict(list)
        
        main_candidate_embs = defaultdict(list)
        main_vacancy_embs = defaultdict(list)
        context_embs_can = defaultdict(list)
        context_embs_com = defaultdict(list)
        
        # Reference device for zero-padding
        ref_key = next(iter(data.x_dict))
        device = data.x_dict[ref_key].device

        for ntype in data.node_types:
            if ntype in x_can_dict:
                for i, emb in enumerate(x_can_dict[ntype]):
                    u_id = data[ntype].unique_node_id[i].item()
                    if u_id == 0: continue # Skip dummy
                    
                    sg = int(data[ntype].sub_graph[i].item())
                    
                    # Check if this specific node is a 'Main' node
                    is_head = u_id in data.head_nodes
                    is_tail = u_id in data.tail_nodes

                    if is_head:
                        main_candidate_embs[sg].append(emb)
                    if is_tail:
                        main_vacancy_embs[sg].append(emb)
                    
                    # All nodes (including head/tail) contribute to sub-graph context
                    context_embs_can[sg].append(emb.unsqueeze(0))
                    context_embs_com[sg].append(x_com_dict[ntype][i].unsqueeze(0))

        # Final Vector Construction
        final_can_list, final_com_list = [], []
        
        # We iterate through all sub-graphs present in this batch
        all_sgs = sorted(context_embs_can.keys())
        for sg in all_sgs:
            # Pool the general graph context (size: embedding_size)
            pooled_can_ctx = self.pooling(torch.stack(context_embs_can[sg]).squeeze(1), dim=0)
            pooled_com_ctx = self.pooling(torch.stack(context_embs_com[sg]).squeeze(1), dim=0)

            # Get ONE Candidate embedding (Mean pool if multiple found, zero if none)
            if main_candidate_embs[sg]:
                can_main = torch.mean(torch.stack(main_candidate_embs[sg]), dim=0)
            else:
                can_main = torch.zeros(self.embedding_size, device=device)

            # Get ONE Vacancy embedding (Mean pool if multiple found, zero if none)
            if main_vacancy_embs[sg]:
                vac_main = torch.mean(torch.stack(main_vacancy_embs[sg]), dim=0)
            else:
                vac_main = torch.zeros(self.embedding_size, device=device)

            # Construct fixed-size vectors (Always size 3 * embedding_size)
            # Order: [Candidate_Node, Vacancy_Node, Subgraph_Context]
            can_vec = torch.cat([can_main, vac_main, pooled_can_ctx])
            com_vec = torch.cat([can_main, vac_main, pooled_com_ctx]) # Using main nodes from com side if desired

            final_can_list.append(can_vec)
            final_com_list.append(com_vec)

        # Stack and Predict
        can_matrix = torch.stack(final_can_list, dim=0)
        com_matrix = torch.stack(final_com_list, dim=0)
                
        # Make predictions based on the sub-graph embeddings
        y_candidate = torch.clamp(self.mlp_candidate(can_matrix), min=-100, max=100)
        y_company = torch.clamp(self.mlp_company(com_matrix), min=-100, max=100)
        
        # Predicted score (Harmonic Mean)
        y_pred = torch.nan_to_num(2 * ((y_candidate * y_company) / (y_candidate + y_company + 1e-8))).squeeze()
        
        return y_pred, y_candidate, y_company, None

In [8]:
def train_loop(model, optimizer, trainloader, valloader, epochs=10, patience=4, kind="val"):
    ndcg_scores = []
    random_scores = []
    
    best_ndcg = -float('inf')
    patience_counter = 0
    
    # 1. Ensure optimizer is looking at the LATEST model parameters
    # (Especially important if you used to_hetero recently)
    optimizer.param_groups[0]['params'] = list(model.parameters())

    for epoch in range(epochs):
        model.train()
        for i, data in enumerate(trainloader):
            data = data.to(device)
            optimizer.zero_grad() 

            # Forward Pass
            y_pred, _, _, _ = model(data)

            data = data.to(device)
            optimizer.zero_grad() # Clear old gradients

            # Forward Pass
            y_pred, _, _, _ = model(data)

            # Calculate Gradient (lambda_i)
            lambda_i = listwise_loss(y_pred, data.y)
            torch.autograd.backward(y_pred.view(-1), lambda_i.view(-1))

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            # Logging
            y_true = data.y.view(1, -1).cpu() 
            y_score = y_pred.detach().view(1, -1).cpu()
            
            # Use k=min(10, len(y_true)) to avoid errors on small batches
            batch_ndcg = ndcg_score(y_true, y_score, k=min(10, y_true.shape[1]))
            ndcg_scores.append(batch_ndcg)
            
            random_y = torch.rand_like(data.y).view(1, -1).cpu()
            random_scores.append(ndcg_score(y_true, random_y, k=min(10, y_true.shape[1])))

            print(" " * 100, end="\r")
            print(f"Epoch: {epoch + 1}, Batch: {i}/{len(trainloader)}, Pred Mean: {y_pred.mean().item():.4f}, NDCG: {batch_ndcg:.4f}", end="\r")

        print(f"\n\nTraining nDCG: {np.mean(ndcg_scores):.4f}")
        print(f"Training random nDCG: {np.mean(random_scores):.4f}\n")

        ndcg_scores = []
        random_scores = []
        
        # Evaluate model
        ndcg_val, random_scores_val = eval_loop(model, valloader, kind=kind)
        ndcg_outcome = np.mean(ndcg_val)
        
        print(f"\n{'Validation' if kind=='val' else 'Test'} nDCG: {np.mean(ndcg_val):.4f}")
        print(f"{'Validation' if kind=='val' else 'Test'}  random nDCG: {np.mean(random_scores_val):.4f}\n")

        # Check for improvement
        if ndcg_outcome > best_ndcg * 1:
            best_ndcg = ndcg_outcome
            best_epoch = epoch + 1
            patience_counter = 0
            # Save the best model state
            print(f"--> Improvement! Best nDCG: {best_ndcg:.4f}")
        else:
            patience_counter += 1
            print(f"--> No improvement. Patience: {patience_counter}/{patience}")

        # Stop if we haven't improved for 'patience' epochs
        if patience_counter >= patience:
            print(f"\nEarly stopping triggered at epoch {epoch + 1}")
            break
        
    return best_ndcg, best_epoch

def eval_loop(model, valloader, kind="val"):
    model.eval()
    ndcg_scores = []
    random_scores = []

    with torch.no_grad():
        for i, data_val in enumerate(valloader):
            data_val = data_val.to(device)
            print(f"Batch ({kind}): {i + 1}/{len(valloader)}", end="\r")
            
            y_pred_val, _, _, _ = model(data_val)
             
            y_true = data_val.y.unsqueeze(0).cpu()
            y_score = y_pred_val.unsqueeze(0).cpu()

            ndcg_scores.append(ndcg_score(y_true, y_score, k=10))
            random_scores.append(ndcg_score(y_true, torch.rand_like(data_val.y).unsqueeze(0).cpu(), k=10))
            
    return ndcg_scores, random_scores

In [9]:
def optimize_model(trial, trainloader, valloader, epochs=10, patience=4):
    # Grab an example batch to get the metadata for the HeteroGNN structure
    example_batch = next(iter(trainloader))

    # Cleaned up search space (No textual params)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
    embedding_size = trial.suggest_categorical('embedding_size', [8, 16, 32, 64, 128])
    pooling_method = trial.suggest_categorical('pooling_method', ["mean", "max", "sum"])
    heads = trial.suggest_categorical('heads', [2, 4, 8])
                                        
    print(f"""
    Config:
    - learning_rate = {learning_rate}
    - embedding_size = {embedding_size}
    - pooling_method = {pooling_method}
    - heads = {heads}
    """)

    model = OKRA(
        metadata=example_batch.metadata(),
        embedding_size=embedding_size,
        pooling_method=pooling_method,
        heads=heads
    ).to(device)

    # Configure Optimizer
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    
    start_time = time.time() 
    
    # Train and evaluate model
    ndcg_scores_val, best_epoch = train_loop(model, optimizer, trainloader, valloader, epochs=epochs, patience=patience)
    
    trial.set_user_attr("epochs", best_epoch)
    
    end_time = time.time()
    
    print(f"Training for {epochs} epochs took {end_time - start_time:.2f} seconds")
    
    return np.mean(ndcg_scores_val)

In [10]:
def objective_wrapper(trainloader, valloader):
    def objective(trial):
        return optimize_model(trial, trainloader, valloader, epochs=15)
    
    return objective

In [12]:
results = defaultdict(list)

if "OKRA_validation.csv" in os.listdir():
    done = pd.read_csv("OKRA_validation.csv")
    
    completed_runs = set(
            zip(done['inference'].astype(bool), 
                done['isco'].astype(bool), 
                done['model'].astype(str), 
                done['prompt'].astype(str))
        )
    print(f"Resuming... Found {len(completed_runs)} completed configurations.")
else:
    completed_runs = set()

for inference in [True, False]:
    for isco in [True, False]:
        for model in ["qwen", "gemma", "llama"]:
            for prompt in ["structured", "semi-structured", "unstructured"]:
                # Check if this combination was already evaluated
                if (inference, isco, model, prompt) in completed_runs:
                    print(f"Skipping {model} {prompt} with inference={inference} and isco={isco} (Already done)")
                    continue
                
                if inference:
                    if isco:
                        trainloader = torch.load(f'../dataloaders/graph_trainloader_{model}_{prompt}_isco.pth',
                                                 weights_only=False)
                        valloader = torch.load(f'../dataloaders/graph_valloader_{model}_{prompt}_isco.pth',
                                               weights_only=False)
                    else:
                        trainloader = torch.load(f'../dataloaders/graph_trainloader_{model}_{prompt}.pth',
                                                 weights_only=False)
                        valloader = torch.load(f'../dataloaders/graph_valloader_{model}_{prompt}.pth',
                                               weights_only=False)
                else:
                    if isco:
                        trainloader = torch.load(f'../dataloaders/{model}_{prompt}_isco_trainloader_no_inference.pth',
                                                 weights_only=False)
                        valloader = torch.load(f'../dataloaders/{model}_{prompt}_isco_valloader_no_inference.pth',
                                               weights_only=False)
                    else:
                        trainloader = torch.load(f'../dataloaders/{model}_{prompt}_trainloader_no_inference.pth',
                                                 weights_only=False)
                        valloader = torch.load(f'../dataloaders/{model}_{prompt}_valloader_no_inference.pth',
                                               weights_only=False)
                    
                torch.cuda.empty_cache() 
                gc.collect()
                
                # Hide user/future warnings
                warnings.filterwarnings('ignore')
                
                # Define the Optuna study
                study = optuna.create_study(direction='maximize')
                print(f"Running study for {model} {prompt} with inference={inference} and isco={isco}")
                
                # We need to provide trainloader and valloader to the training/validation loop
                wrapped_objective = objective_wrapper(trainloader, valloader)
                
                # Start optimization
                study.optimize(wrapped_objective, n_trials=3)  
                
                print("Best hyperparameters:", study.best_trial.params)

                results["model"].append(model)
                results["prompt"].append(prompt)
                results["inference"].append(inference)
                results["isco"].append(isco)
                
                for param in study.best_trial.params:
                    results[param].append(study.best_trial.params[param])

                results["epochs"].append(study.best_trial.user_attrs["epochs"])

                results["nDCG@10 (val)"].append(study.best_value)
                    
                df_results = pd.DataFrame(results)
                display(df_results)

df_results.to_csv("OKRA_validation.csv")

[I 2026-06-08 01:20:47,596] A new study created in memory with name: no-name-8512ef6a-e265-41b3-b80c-9b5c2471de81


Running study for qwen structured with inference=True and isco=True

    Config:
    - learning_rate = 0.0004382333856990665
    - embedding_size = 128
    - pooling_method = max
    - heads = 4
    
Epoch: 1, Batch: 288/289, Pred Mean: -0.0373, NDCG: 0.7039                                          

Training nDCG: 0.6778
Training random nDCG: 0.3569

Batch (val): 41/41
Validation nDCG: 0.7695
Validation  random nDCG: 0.3831

--> Improvement! Best nDCG: 0.7695
Epoch: 2, Batch: 288/289, Pred Mean: 0.3304, NDCG: 0.8711                                           

Training nDCG: 0.7567
Training random nDCG: 0.3940

Batch (val): 41/41
Validation nDCG: 0.7819
Validation  random nDCG: 0.4433

--> Improvement! Best nDCG: 0.7819
Epoch: 3, Batch: 288/289, Pred Mean: 0.5831, NDCG: 0.9060                                           

Training nDCG: 0.7959
Training random nDCG: 0.3573

Batch (val): 41/41
Validation nDCG: 0.7619
Validation  random nDCG: 0.3141

--> No improvement. Patience: 1/4
Epoch:

[I 2026-06-08 01:27:55,720] Trial 0 finished with value: 0.8087630974071186 and parameters: {'learning_rate': 0.0004382333856990665, 'embedding_size': 128, 'pooling_method': 'max', 'heads': 4}. Best is trial 0 with value: 0.8087630974071186.


Batch (val): 41/41
Validation nDCG: 0.7871
Validation  random nDCG: 0.3056

--> No improvement. Patience: 2/4
Training for 15 epochs took 427.91 seconds

    Config:
    - learning_rate = 0.0012856154878068266
    - embedding_size = 8
    - pooling_method = sum
    - heads = 2
    
Epoch: 1, Batch: 288/289, Pred Mean: 0.0552, NDCG: 0.6797                                           

Training nDCG: 0.6124
Training random nDCG: 0.3854

Batch (val): 41/41
Validation nDCG: 0.7671
Validation  random nDCG: 0.4220

--> Improvement! Best nDCG: 0.7671
Epoch: 2, Batch: 288/289, Pred Mean: 0.7287, NDCG: 0.7328                                           

Training nDCG: 0.7808
Training random nDCG: 0.3703

Batch (val): 41/41
Validation nDCG: 0.7820
Validation  random nDCG: 0.3941

--> Improvement! Best nDCG: 0.7820
Epoch: 3, Batch: 288/289, Pred Mean: 1.0623, NDCG: 0.7328                                           

Training nDCG: 0.8064
Training random nDCG: 0.3672

Batch (val): 41/41
Validation nDC

[I 2026-06-08 01:34:07,381] Trial 1 finished with value: 0.8082293714244813 and parameters: {'learning_rate': 0.0012856154878068266, 'embedding_size': 8, 'pooling_method': 'sum', 'heads': 2}. Best is trial 0 with value: 0.8087630974071186.


...

[I 2026-06-09 03:47:13,358] Trial 1 finished with value: 0.7879744444025734 and parameters: {'learning_rate': 0.0006217308603403987, 'embedding_size': 64, 'pooling_method': 'mean', 'heads': 8}. Best is trial 1 with value: 0.7879744444025734.



Validation nDCG: 0.7621
Validation  random nDCG: 0.3392

--> No improvement. Patience: 4/4

Early stopping triggered at epoch 13
Training for 15 epochs took 3123.42 seconds

    Config:
    - learning_rate = 0.0007506733115176721
    - embedding_size = 16
    - pooling_method = mean
    - heads = 4
    
Epoch: 1, Batch: 288/289, Pred Mean: 0.0349, NDCG: 0.6714                                           

Training nDCG: 0.6800
Training random nDCG: 0.3597

Batch (val): 41/41
Validation nDCG: 0.7738
Validation  random nDCG: 0.4045

--> Improvement! Best nDCG: 0.7738
Epoch: 2, Batch: 288/289, Pred Mean: 0.1572, NDCG: 0.6257                                           

Training nDCG: 0.7816
Training random nDCG: 0.3673

Batch (val): 41/41
Validation nDCG: 0.8003
Validation  random nDCG: 0.3264

--> Improvement! Best nDCG: 0.8003
Epoch: 3, Batch: 288/289, Pred Mean: 0.2178, NDCG: 0.7654                                           

Training nDCG: 0.7876
Training random nDCG: 0.3601

Batch (val

[I 2026-06-09 04:11:22,911] Trial 2 finished with value: 0.8002549589273851 and parameters: {'learning_rate': 0.0007506733115176721, 'embedding_size': 16, 'pooling_method': 'mean', 'heads': 4}. Best is trial 2 with value: 0.8002549589273851.



Validation nDCG: 0.7705
Validation  random nDCG: 0.3158

--> No improvement. Patience: 4/4

Early stopping triggered at epoch 6
Training for 15 epochs took 1449.47 seconds
Best hyperparameters: {'learning_rate': 0.0007506733115176721, 'embedding_size': 16, 'pooling_method': 'mean', 'heads': 4}


,model,prompt,inference,isco,learning_rate,embedding_size,pooling_method,heads,epochs,nDCG@10 (val)
0,qwen,structured,True,True,0.000438,128,max,4,13,0.808763
1,qwen,semi-structured,True,True,0.000238,128,mean,2,9,0.792956
2,qwen,unstructured,True,True,0.001033,16,max,2,9,0.812563
3,gemma,structured,True,True,0.000282,64,mean,2,5,0.814214
4,gemma,semi-structured,True,True,0.004548,128,sum,2,3,0.885389
5,gemma,unstructured,True,True,0.007879,8,max,4,4,0.841647
6,llama,structured,True,True,0.009736,16,mean,8,6,0.858868
7,llama,semi-structured,True,True,0.005019,64,sum,2,2,0.840683
8,llama,unstructured,True,True,0.000114,128,max,2,3,0.698263
9,qwen,structured,True,False,0.003904,128,sum,8,12,0.853009


In [13]:
df_results = pd.read_csv("OKRA_validation.csv")

runs_to_do = set(
        zip(df_results['inference'].astype(bool), 
            df_results['isco'].astype(bool), 
            df_results['model'].astype(str), 
            df_results['prompt'].astype(str))
    )
print(f"Resuming... Found {len(completed_runs)} completed configurations.")

Resuming... Found 0 completed configurations.


# Evaluation 

In [14]:
test_results = defaultdict(list)

for inference in [True, False]:
    for isco in [True, False]:
        for model in ["qwen", "gemma", "llama"]:
            for prompt in ["structured", "semi-structured", "unstructured"]:

                # Check if this combination was already evaluated
                if (inference, isco, model, prompt) not in runs_to_do:
                    print(f"Skipping {model} {prompt} with inference={inference} and isco={isco} (Already done)")
                    continue
                
                
                if inference:
                    if isco:
                        trainloader = torch.load(f'../dataloaders/graph_trainloader_{model}_{prompt}_isco.pth',
                                                 weights_only=False)
                        testloader = torch.load(f'../dataloaders/graph_testloader_{model}_{prompt}_isco.pth',
                                               weights_only=False)
                    else:
                        trainloader = torch.load(f'../dataloaders/graph_trainloader_{model}_{prompt}.pth',
                                                 weights_only=False)
                        testloader = torch.load(f'../dataloaders/graph_testloader_{model}_{prompt}.pth',
                                               weights_only=False)
                else:
                    if isco:
                        trainloader = torch.load(f'../dataloaders/{model}_{prompt}_isco_trainloader_no_inference.pth',
                                                 weights_only=False)
                        testloader = torch.load(f'../dataloaders/{model}_{prompt}_isco_testloader_no_inference.pth',
                                               weights_only=False)
                    else:
                        trainloader = torch.load(f'../dataloaders/{model}_{prompt}_trainloader_no_inference.pth',
                                                 weights_only=False)
                        testloader = torch.load(f'../dataloaders/{model}_{prompt}_testloader_no_inference.pth',
                                               weights_only=False)

                optimal_parameters = df_results[(df_results["model"] == model) & 
                    (df_results["prompt"] == prompt) &
                    (df_results["inference"] == inference) & 
                    (df_results["isco"] == isco)].iloc[0].values

                learning_rate = float(optimal_parameters[5])
                embedding_size = int(optimal_parameters[6])
                pooling_method = optimal_parameters[7]
                heads = int(optimal_parameters[8])
                epochs = int(optimal_parameters[9])
                
                print(f"""
                Config:
                - learning_rate = {learning_rate}
                - embedding_size = {embedding_size}
                - pooling_method = {pooling_method}
                - heads = {heads}
                """)

                example_batch = next(iter(trainloader))

                okra = OKRA(
                    metadata=example_batch.metadata(),
                    embedding_size=embedding_size,
                    pooling_method=pooling_method,
                    heads=heads
                ).to(device)
            
                # Configure Optimizer
                optimizer = torch.optim.Adam(okra.parameters(), lr=learning_rate)
                
                start_time = time.time() 
                
                # Train and test model
                ndcg_scores_test = train_loop(okra, optimizer, trainloader, testloader,
                                              epochs=epochs, patience=99, kind="test")

                test_results["model"].append(model)
                test_results["prompt"].append(prompt)
                test_results["inference"].append(inference)
                test_results["isco"].append(isco)
                test_results["nDCG@10 (test)"].append(ndcg_scores_test[0])

df_test_results = pd.DataFrame(test_results)
df_test_results.to_csv("OKRA_test.csv")
display(df_test_results)


                Config:
                - learning_rate = 0.000438233385699
                - embedding_size = 128
                - pooling_method = max
                - heads = 4
                
Epoch: 1, Batch: 288/289, Pred Mean: 0.0027, NDCG: 1.0000                                           

Training nDCG: 0.6656
Training random nDCG: 0.3613

Batch (test): 37/37
Test nDCG: 0.7461
Test  random nDCG: 0.3548

--> Improvement! Best nDCG: 0.7461
Epoch: 2, Batch: 288/289, Pred Mean: 0.2765, NDCG: 0.8855                                           

Training nDCG: 0.7972
Training random nDCG: 0.3698

Batch (test): 37/37
Test nDCG: 0.7568
Test  random nDCG: 0.3861

--> Improvement! Best nDCG: 0.7568
Epoch: 3, Batch: 288/289, Pred Mean: 0.1891, NDCG: 0.8855                                           

Training nDCG: 0.8081
Training random nDCG: 0.3516

Batch (test): 37/37
Test nDCG: 0.7481
Test  random nDCG: 0.3250

--> No improvement. Patience: 1/99
Epoch: 4, Batch: 288/289, Pred Mean: 0

,model,prompt,inference,isco,nDCG@10 (test)
0,qwen,structured,True,True,0.765140
1,qwen,semi-structured,True,True,0.825452
2,qwen,unstructured,True,True,0.776981
3,gemma,structured,True,True,0.776130
4,gemma,semi-structured,True,True,0.835118
5,gemma,unstructured,True,True,0.818695
6,llama,structured,True,True,0.772975
7,llama,semi-structured,True,True,0.827648
8,llama,unstructured,True,True,0.800592
9,qwen,structured,True,False,0.773577
